# 🫀 실험 15d — **CI 에 빠져 있던 잡음**: 재학습(시드) 분산을 잰다

**MedKOS / `notebooks/exp15d_seed_variance.ipynb`** · 퀘스트 `ailab-2026-0015`
**다운로드 없음** · 실험15·15c 의 arm 을 **시드 0으로 재사용**하고 **2개 시드만 추가**한다.

---

## 지금까지 모든 CI 가 같은 것을 빠뜨렸다

실험 1~15c 의 부트스트랩은 전부 **테스트셋 재표본** 잡음만 쟀다.
그런데 실험15c 가 비교한 것은 **두 학습 절차**(공유 헤드 vs 개별 헤드)였고
`n_seeds = 1` 이었다. **재학습 잡음이 CI 에 아예 없다.**

실험15c 에서 유일하게 유의했던 값이 `IMI {12}` Δ = **−0.0301** [−0.0455, −0.0144] 인데,
효과가 0.02~0.03 규모다. **재학습 잡음이 그만큼이면 이 유의성은 사라진다.**
지금은 그게 얼마인지 **모른다** — 재본 적이 없기 때문이다.

## 그래서 이 실험은 결과가 아니라 **계측기의 오차막대를 잰다**

| 잡음의 출처 | 지금까지 | 이 실험 |
|---|---|---|
| 테스트셋 재표본 | ✅ 부트스트랩으로 잼 | 그대로 |
| **재학습(시드)** | ❌ **안 잼** | **잰다** |
| 같은 시드의 비결정성(GPU) | ❌ 안 잼 | **G0 에서 확인** |

산출물은 숫자 하나다:

> **단일 시드 실험의 CI 는 실제보다 몇 배 좁은가** = `√(1 + (SD_시드 / SD_테스트)²)`

이 값이 1에 가까우면 지금까지의 CI 를 그대로 믿어도 되고, 크면 **실험 1~15c 의 모든
CI 에 단서를 붙이고 앞으로는 시드를 필수로** 돌려야 한다.

## 무엇을 얼마나 돌리나 — 시드 0은 **이미 있다**

실험15(공유)와 15c(개별) 모두 `build_head(SEED0 + 100*k + 15 + 0)` 으로 학습했다.
**그게 시드 0**이므로 읽어 쓰고, **시드 1·2 만 새로 학습**한다.

| | 새로 학습 |
|---|---|
| 공유 7-way 헤드 `{12}` | 2 시드 × 5 겹 = 10 |
| 개별 `IMI` `{12}` | 10 |
| 개별 `LMI` `{12}` | 10 |
| **합** | **30회 (~35분)** |

공유 헤드는 7부위를 함께 내므로 **7부위 전부의 시드 분산이 공짜로** 따라온다.

**`{12}` 만 보는 이유**: 실험15c 의 신호가 거기 있었다(`IMI` −0.0301 ✗ · `LMI` +0.0257 ·
P-3 +0.0440 ✅). 잡음을 재는 데 두 구성이 필요하지 않다.

## 사전등록 (결과 보기 전에 고정)

| | 예측 | 성격 |
|---|---|---|
| **G0 결정론** | 시드 0을 겹 0에서 다시 학습하면 저장된 arm 을 **재현**한다(AUPRC 차 < 0.005) | 재현 안 되면 **같은 시드에도 잡음이 있다**는 뜻이고, 그것부터 보고한다 |
| **P-1 ★** | `SD_시드 ≥ 0.5 × SD_테스트` 인 칸이 하나라도 있다 | **시드 잡음이 무시할 수 없다** → 앞으로 CI 에 넣는다 |
| **P-2** | `IMI {12}` 의 Δ 가 **3 시드 모두 음수** | 15c 의 유일한 유의 결과가 시드를 넘어 남는가 |
| **P-3** | `LMI {12}` 의 Δ 가 **3 시드 모두 양수** | 희소 부위의 공유 이득이 시드를 넘어 남는가 |

> ⚠️ **P-2·P-3 의 부호 일치는 약한 검정이다.** 효과가 없어도 3개가 같은 부호일 확률이
> 2/8 = **0.25** 다. 그래서 **시드 수준 평균의 t-CI(df=2)** 도 함께 낸다 — 매우 넓겠지만
> 그게 정직한 폭이다. 판정은 `decide()` 로 3분한다.

## 이 실험이 무엇을 정하나

- `P-1` ✅ → **실험 1~15c 의 CI 에 "재학습 잡음 미포함" 단서를 붙이고**, 앞으로 결론에
  쓰는 실험은 **시드 ≥ 3** 을 기본으로 한다. 필요한 시드 수도 여기서 계산해 나온다.
- `P-1` ❌ → 지금까지의 CI 를 그대로 쓴다. 시드는 선택 사항이 된다.
- `P-2` ❌ → **실험15c 의 `IMI {12}` 유의성을 철회**하고 로그에 소급 표기한다.


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/ecg_preflight.py 인라인)
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 함수 적재: assert_label_vocab · decide · boot_indices")

In [ ]:
# CELL 1 — 설정 + 실험15·15c 산출물 연결
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15·15c 와 한 글자도 달라선 안 되는 블록
CONFIGS = {"I+II": [0, 1], "12": list(range(12))}
K_FOLD, EPOCHS, SEED0, BOOT, NMIN = 5, 20, 20260801, 2000, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
# ★★ 여기까지

CFG   = "12"                 # 실험15c 의 신호가 있던 구성
SEEDS = [0, 1, 2]            # 0 은 실험15·15c 에서 재사용
SOLO  = ["IMI", "LMI"]       # 풍부 1 + 희소 1 (15c 에서 방향이 갈린 두 끝)
RATIO_THR = 0.5              # P-1: SD_시드 / SD_테스트 문턱

CONFIG = dict(exp="exp15d_seed_variance", quest="ailab-2026-0015",
              parent_exp=["exp15_mi_loc_head", "exp15c_shared_vs_solo"],
              purpose="지금까지 모든 CI 에서 빠져 있던 재학습(시드) 잡음을 잰다",
              deliverable="단일 시드 CI 의 과소평가 배수 = sqrt(1 + (SD_seed/SD_test)^2)",
              config=CFG, seeds=SEEDS, solo_sites=SOLO,
              reuse="시드 0 은 실험15(공유)·15c(개별)에서 읽어 쓴다 — 30회만 새로 학습",
              predictions={"G0": "시드 0 재학습이 저장된 arm 을 재현(AUPRC 차 < 0.005)",
                           "P-1": f"SD_seed >= {RATIO_THR} x SD_test 인 칸이 하나라도",
                           "P-2": "IMI {12} 의 Δ 가 3 시드 모두 음수",
                           "P-3": "LMI {12} 의 Δ 가 3 시드 모두 양수"},
              caveat="3 시드의 부호 일치는 귀무 하에서도 확률 0.25 — t-CI(df=2)도 함께 낸다",
              k_fold=K_FOLD, epochs=EPOCHS, seed0=SEED0, boot=BOOT)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp15d_seed_var", CONFIG, project=PROJECT)

REG = os.path.join(PROJECT, "registry.jsonl")
DIRS = {}
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") in ("exp15_mi_loc_head", "exp15c_shared_vs_solo") \
            and os.path.isdir(r.get("dir", "")):
        DIRS[r["exp_id"]] = r["dir"]
for k in ("exp15_mi_loc_head", "exp15c_shared_vs_solo"):
    if k not in DIRS:
        raise RuntimeError(f"registry.jsonl 에서 {k} 를 못 찾았습니다")
P15, P15C = DIRS["exp15_mi_loc_head"], DIRS["exp15c_shared_vs_solo"]
run.log(f"실험15 : {P15}")
run.log(f"실험15c: {P15C}")
SITES15 = json.load(open(os.path.join(P15, "result.json"), encoding="utf-8"))["sites"]
run.log(f"  실험15 부위 {SITES15}")

def arm_at(d, name):
    p = os.path.join(d, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

need = [(P15, f"{CFG}_f{k}") for k in range(K_FOLD)] + \
       [(P15C, f"solo_{s}_{CFG}_f{k}") for s in SOLO for k in range(K_FOLD)]
missing = [n for d, n in need if arm_at(d, n) is None]
if missing:
    raise RuntimeError(f"시드 0 arm 없음: {missing[:5]} …")
run.log(f"✅ 시드 0 arm {len(need)}개 확인 — 새로 학습할 것은 시드 {SEEDS[1:]} 뿐")

In [ ]:
# CELL 2 — 캐시 재사용 + 라벨 (다운로드 없음)
import pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
for f in ("ptbxl_database.csv",):
    d = os.path.join(PTB, f)
    if not (os.path.exists(d) and os.path.getsize(d) > 0):
        subprocess.run(["wget", "-q", "-O", d, f"{BASE}/{f}"])
df = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")

CACHE = run.data("ptbxl_12lead_all.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"전량 캐시가 없습니다: {CACHE} — 실험15 CELL 2 를 먼저")
z = np.load(CACHE, allow_pickle=True)
X, FOLD10, EID = z["X"], z["fold"], z["eid"]
CV = (FOLD10 - 1) % K_FOLD
dfa = df.loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
vocab = {c for cs in dfa.codes for c in cs}
counts = {s: int(sum(s in cs for cs in dfa.codes)) for s in SITE_CANDIDATES}
assert_label_vocab(SITE_CANDIDATES, vocab, kind="scp 코드", counts=counts, min_count=NMIN)
SITES = [s for s in SITE_CANDIDATES if counts[s] >= NMIN]
if SITES != SITES15:
    raise RuntimeError(f"부위 목록이 실험15와 다릅니다: {SITES} vs {SITES15}")
Ymul = np.stack([[s in c for s in SITES] for c in dfa.codes]).astype("float32")
run.log(f"캐시 재사용 X{X.shape} · 부위 {len(SITES)}개 · 정렬 일치 ✅")

MK = np.zeros(12, "float32"); MK[CONFIGS[CFG]] = 1.0

In [ ]:
# CELL 3 — 【G0】 결정론 확인 + 시드 1·2 학습 (30회)
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import average_precision_score

def build_head(seed, n_out):
    """★ 실험15·15c 와 완전히 동일. 출력 차원만 인자로 받는다."""
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], 12))
    x = si
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(si, layers.Dense(n_out, activation="sigmoid")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="binary_crossentropy")
    return m

def split(k):
    te = np.where(CV == k)[0]; rest = np.where(CV != k)[0]
    rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
    n_val = max(int(len(rest) * 0.12), 200)
    return te, rest[:n_val], rest[n_val:]

def train_fold(k, sd, Y, n_out):
    te, va, tr = split(k)
    m = build_head(SEED0 + 100 * k + 15 + sd, n_out)     # ★ 실험15·15c 와 같은 공식
    m.fit(X[tr] * MK, Y[tr], validation_data=(X[va] * MK, Y[va]),
          epochs=EPOCHS, batch_size=128, verbose=0)
    p = m.predict(X[te] * MK, batch_size=512, verbose=0)
    tf.keras.backend.clear_session()
    return te, p

# ── 【G0】 시드 0·겹 0 을 다시 학습해 저장된 arm 을 재현하는가
run.log("\n【G0】 결정론 확인 — 시드 0·겹 0 재학습")
te0, p0 = train_fold(0, 0, Ymul, len(SITES))
stored = arm_at(P15, f"{CFG}_f0")[te0]
j_imi = SITES.index("IMI")
y0 = Ymul[te0, j_imi].astype(bool)
ap_new = float(average_precision_score(y0, p0[:, j_imi]))
ap_old = float(average_precision_score(y0, stored[:, j_imi]))
maxabs = float(np.abs(p0 - stored).max())
run.log(f"  IMI AUPRC 재학습 {ap_new:.4f} vs 저장 {ap_old:.4f} · 차 {ap_new-ap_old:+.4f}")
run.log(f"  확률 최대 절대차 {maxabs:.2e}")
G0 = bool(abs(ap_new - ap_old) < 0.005)
run.log("  " + ("✅ 재현 — 같은 시드는 같은 결과" if G0 else
                "⚠️ 재현 안 됨 — **같은 시드에도 잡음이 있다**(GPU 비결정성). "
                "아래 시드 분산에 이 성분이 섞인다는 것을 감안해 읽을 것"))

# ── 시드 1·2 학습
SH = {0: np.zeros((len(EID), len(SITES)), "float32")}
for k in range(K_FOLD):
    SH[0][np.where(CV == k)[0]] = arm_at(P15, f"{CFG}_f{k}")
SO = {s: {0: np.zeros(len(EID), "float32")} for s in SOLO}
for s in SOLO:
    for k in range(K_FOLD):
        SO[s][0][np.where(CV == k)[0]] = arm_at(P15C, f"solo_{s}_{CFG}_f{k}").ravel()

t0, done, total = time.time(), 0, (1 + len(SOLO)) * (len(SEEDS) - 1) * K_FOLD
for sd in SEEDS[1:]:
    SH[sd] = np.zeros((len(EID), len(SITES)), "float32")
    for k in range(K_FOLD):
        a = run.load_arm(f"shared_s{sd}_f{k}")
        if a is None:
            te, p = train_fold(k, sd, Ymul, len(SITES)); run.save_arm(f"shared_s{sd}_f{k}", p)
            a = p
        else:
            te = np.where(CV == k)[0]
        SH[sd][te] = a; done += 1
        if done == 1:
            per = time.time() - t0
            run.log(f"  ⏱ 첫 학습 {per:.0f}s → 전체 {total}회 예상 **{per*total/60:.0f}분**")
    run.log(f"  공유 시드{sd} 완료 ({done}/{total} · {time.time()-t0:.0f}s)")
    for s in SOLO:
        j = SITES.index(s); SO[s][sd] = np.zeros(len(EID), "float32")
        for k in range(K_FOLD):
            a = run.load_arm(f"solo_{s}_s{sd}_f{k}")
            if a is None:
                te, p = train_fold(k, sd, Ymul[:, j:j + 1], 1)
                run.save_arm(f"solo_{s}_s{sd}_f{k}", p); a = p
            else:
                te = np.where(CV == k)[0]
            SO[s][sd][te] = a.ravel(); done += 1
        run.log(f"  개별 {s} 시드{sd} 완료 ({done}/{total} · {time.time()-t0:.0f}s)")
run.log(f"\n총 {time.time()-t0:.0f}s")

In [ ]:
# CELL 4 — 분산 분해: 테스트셋 잡음 vs 재학습 잡음
q = lambda v, p: float(np.percentile(v, p))

def boot_sd(a, b, y, seed=SEED0):
    """Δ(AUPRC) 의 테스트셋 부트스트랩 분포."""
    out = np.full(BOOT, np.nan)
    for t, i in enumerate(boot_indices(len(y), BOOT, seed)):
        if y[i].sum() < 5:
            continue
        out[t] = (average_precision_score(y[i], a[i])
                  - average_precision_score(y[i], b[i]))
    v = out[~np.isnan(out)]
    return float(v.mean()), float(v.std(ddof=1)), q(v, 2.5), q(v, 97.5)

run.log("\n" + "=" * 112)
run.log("【시드별 Δ(공유 − 개별)】 · 구성 {12}")
run.log("=" * 112)
run.log(f"  {'부위':<6}{'시드':>5}{'공유 AUPRC':>12}{'개별 AUPRC':>12}{'Δ':>10}"
        f"{'테스트 SD':>11}{'   테스트셋 95% CI':<22}")
VAR = {}
for s in SOLO:
    j = SITES.index(s); y = Ymul[:, j].astype(bool)
    per_seed = []
    for sd in SEEDS:
        sh, so = SH[sd][:, j], SO[s][sd]
        ap_sh = float(average_precision_score(y, sh))
        ap_so = float(average_precision_score(y, so))
        d, sdt, lo, hi = boot_sd(sh, so, y)
        per_seed.append({"seed": sd, "auprc_shared": ap_sh, "auprc_solo": ap_so,
                         "delta": ap_sh - ap_so, "sd_test": sdt, "ci": [lo, hi]})
        run.log(f"  {s:<6}{sd:>5}{ap_sh:>12.4f}{ap_so:>12.4f}{ap_sh-ap_so:>+10.4f}"
                f"{sdt:>11.4f}   [{lo:+.4f}, {hi:+.4f}]")
    ds = np.array([x["delta"] for x in per_seed])
    sd_seed = float(ds.std(ddof=1))
    sd_test = float(np.mean([x["sd_test"] for x in per_seed]))
    ratio = sd_seed / sd_test if sd_test > 0 else float("inf")
    infl = float(np.sqrt(1 + ratio ** 2))
    # 시드 수준 t-CI (df = n-1). 3 시드면 t(0.975, 2) = 4.303
    t975 = 4.302652729911275
    se = sd_seed / np.sqrt(len(ds))
    t_lo, t_hi = float(ds.mean() - t975 * se), float(ds.mean() + t975 * se)
    VAR[s] = {"per_seed": per_seed, "mean_delta": float(ds.mean()),
              "sd_seed": sd_seed, "sd_test": sd_test, "ratio": ratio,
              "inflation": infl, "t_ci": [t_lo, t_hi],
              "signs": [int(np.sign(d)) for d in ds]}
    run.log(f"  {'':<6}{'—':>5}{'':>12}{'':>12}{ds.mean():>+10.4f}"
            f"{'':>11}   시드 SD {sd_seed:.4f} · 비 {ratio:.2f}")
    run.log(f"         시드 수준 t-CI(df=2): [{t_lo:+.4f}, {t_hi:+.4f}]"
            f"  ← 3점으로 만든 정직한 폭")
    run.log("  " + "-" * 108)

# ── 공유 헤드는 7부위를 함께 내므로 전 부위의 시드 분산이 공짜로 나온다
run.log("\n【공유 헤드 자체의 시드 분산】 (부위별 AUPRC, 시드 3개)")
run.log(f"  {'부위':<7}{'n':>6}" + "".join(f"{'시드' + str(sd):>10}" for sd in SEEDS)
        + f"{'SD':>10}{'변동폭':>10}")
SHVAR = {}
for j, s in enumerate(SITES):
    y = Ymul[:, j].astype(bool)
    aps = [float(average_precision_score(y, SH[sd][:, j])) for sd in SEEDS]
    SHVAR[s] = {"auprc": aps, "sd": float(np.std(aps, ddof=1)),
                "range": float(max(aps) - min(aps)), "n": int(y.sum())}
    run.log(f"  {s:<7}{int(y.sum()):>6}" + "".join(f"{a:>10.4f}" for a in aps)
            + f"{np.std(aps, ddof=1):>10.4f}{max(aps)-min(aps):>10.4f}")

In [ ]:
# CELL 5 — 사전등록 채점 + 앞으로의 규칙
run.log("\n" + "=" * 112)
run.log("【사전등록 채점】")
run.log("=" * 112)
run.log(f"  G0 결정론 → {'✅' if G0 else '⚠️ 같은 시드에도 잡음 있음'}")

worst = max(VAR, key=lambda s: VAR[s]["ratio"])
P1 = bool(any(VAR[s]["ratio"] >= RATIO_THR for s in VAR))
run.log(f"  P-1 ★ SD_시드 ≥ {RATIO_THR}×SD_테스트 인 칸 존재 → {MARK[P1]}")
for s in VAR:
    v = VAR[s]
    run.log(f"      {s:<6} SD_시드 {v['sd_seed']:.4f} / SD_테스트 {v['sd_test']:.4f}"
            f" = {v['ratio']:.2f}  → 단일 시드 CI 는 실제보다 **{v['inflation']:.2f}배 좁다**")

sg_imi = VAR["IMI"]["signs"]
P2 = decide(*VAR["IMI"]["t_ci"], 0.0, "<")
run.log(f"  P-2  IMI {{12}} Δ 가 3 시드 모두 음수 → "
        f"{'✅' if all(x < 0 for x in sg_imi) else '❌'} (부호 {sg_imi})")
run.log(f"       t-CI 기준 판정: {MARK[P2]}  "
        f"평균 {VAR['IMI']['mean_delta']:+.4f} [{VAR['IMI']['t_ci'][0]:+.4f}, "
        f"{VAR['IMI']['t_ci'][1]:+.4f}]")
run.log(f"       ※ 실험15c 는 이 자리를 −0.0301 [−0.0455,−0.0144] 로 **유의**하다고 했다")

sg_lmi = VAR["LMI"]["signs"]
P3 = decide(*VAR["LMI"]["t_ci"], 0.0, ">")
run.log(f"  P-3  LMI {{12}} Δ 가 3 시드 모두 양수 → "
        f"{'✅' if all(x > 0 for x in sg_lmi) else '❌'} (부호 {sg_lmi})")
run.log(f"       t-CI 기준 판정: {MARK[P3]}  "
        f"평균 {VAR['LMI']['mean_delta']:+.4f} [{VAR['LMI']['t_ci'][0]:+.4f}, "
        f"{VAR['LMI']['t_ci'][1]:+.4f}]")
run.log("  ※ 3 시드의 부호 일치는 귀무 하에서도 확률 0.25 — t-CI 를 우선해 읽는다")

# ── 앞으로 몇 시드를 돌려야 하나
run.log("\n" + "=" * 112)
run.log("【앞으로의 규칙】 시드 성분이 테스트 잡음의 절반 아래로 내려가려면")
run.log("=" * 112)
NEED = {}
for s in VAR:
    r = VAR[s]["ratio"]
    # n > 4r² 를 만족하는 최소 정수. ceil 을 쓰면 r=0.5 에서 n=1 이 나와
    # 0.5/√1 = 0.5 로 '< 0.5' 를 못 만족한다(경계 버그).
    n_need = int(np.floor(4 * r ** 2)) + 1
    NEED[s] = max(1, n_need)
    run.log(f"  {s:<6} 비 {r:.2f} → 필요한 시드 수 **{NEED[s]}개**  "
            f"(SD_시드/√n < 0.5×SD_테스트)")
rec = max(NEED.values())
if P1:
    rule = (f"**앞으로 결론에 쓰는 실험은 시드 {rec}개 이상**을 돌리고, CI 에 시드 성분을 "
            f"넣는다. 실험 1~15c 의 CI 에는 '재학습 잡음 미포함' 단서를 붙인다")
else:
    rule = ("시드 잡음이 테스트 잡음에 비해 작다. 지금까지의 CI 를 그대로 쓰고 "
            "시드는 선택 사항으로 둔다")
run.log(f"\n▶ {rule}")

if P1 and P2 is None:
    verdict = ("계측기 보정 — 시드 잡음이 무시할 수 없고(P-1), 실험15c 의 IMI 유의성은 "
               "시드를 넣으면 **미결로 내려간다**. 15c 의 그 결론을 소급 완화한다")
elif P1:
    verdict = (f"계측기 보정 — 시드 잡음이 무시할 수 없다(최대 {VAR[worst]['inflation']:.2f}배). "
               f"앞으로 결론에 쓰는 실험은 시드 {rec}개 이상")
else:
    verdict = "지금까지의 CI 를 그대로 신뢰할 수 있다 — 시드 잡음이 작다"
run.log(f"\n▶ {verdict}")
run.log("=" * 112)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
for ax, s in zip(axes, SOLO):
    v = VAR[s]
    xs = np.arange(len(SEEDS))
    ds = [x["delta"] for x in v["per_seed"]]
    er = [[x["delta"] - x["ci"][0] for x in v["per_seed"]],
          [x["ci"][1] - x["delta"] for x in v["per_seed"]]]
    ax.errorbar(xs, ds, yerr=er, fmt="o", capsize=5, label="시드별 Δ (테스트셋 CI)")
    ax.axhspan(v["t_ci"][0], v["t_ci"][1], alpha=.15, color="C1",
               label="시드 수준 t-CI (df=2)")
    ax.axhline(v["mean_delta"], color="C1", lw=1.2)
    ax.axhline(0, c="k", lw=.9)
    ax.set_xticks(xs); ax.set_xticklabels([f"시드 {sd}" for sd in SEEDS])
    ax.set_ylabel("Δ AUPRC (공유 − 개별)")
    ax.set_title(f"{s} · SD_시드/SD_테스트 = {v['ratio']:.2f}", fontsize=10)
    ax.legend(fontsize=8)
plt.tight_layout(); run.save_fig("seed_variance", fig); plt.show()

run.save_json("evaluation", {"variance": VAR, "shared_seed_var": SHVAR,
                             "G0": G0, "P-1": P1, "P-2": P2, "P-3": P3,
                             "seeds_needed": NEED, "rule": rule, "verdict": verdict})

result = {"week": 2, "exp_id": "exp15d_seed_var", "quest": "ailab-2026-0015",
          "task": "지금까지 모든 CI 에서 빠져 있던 재학습(시드) 잡음을 잰다",
          "split": "inter", "metric": "ci_inflation_factor_max",
          "value": round(max(VAR[s]["inflation"] for s in VAR), 4),
          "passed": bool(P1), "date": time.strftime("%Y-%m-%d"),
          "config": CFG, "seeds": SEEDS, "G0_determinism": G0,
          "variance": VAR, "shared_seed_var": SHVAR, "seeds_needed": NEED,
          "P-1": P1, "P-2": P2, "P-3": P3, "rule": rule, "verdict": verdict,
          "summary": (" · ".join(f"{s} SD_시드/SD_테스트 {VAR[s]['ratio']:.2f}"
                                 f"(CI {VAR[s]['inflation']:.2f}배 좁음)" for s in VAR)
                      + f" · 권장 시드 {rec} · P-2 {MARK[P2]} P-3 {MARK[P3]}")}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp15d_seed_variance.ipynb \\
      --quest ailab-2026-0015 --step "exp15d-seed-variance" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

---

## 이 실험은 결과가 아니라 **계측기를 고친다**

지금까지 나온 모든 CI 는 "테스트셋을 다시 뽑았을 때의 흔들림"만 담고 있었다.
여기서 나오는 **과소평가 배수**가 그 CI 들을 어떻게 읽어야 하는지 정한다.

| P-1 | 뜻 | 조치 |
|---|---|---|
| ✅ | 재학습 잡음이 무시할 수 없다 | 실험 1~15c CI 에 **단서 표기** · 앞으로 시드 N개 필수 |
| ❌ | 테스트 잡음이 지배한다 | 지금까지의 CI 를 그대로 쓴다 |

**P-2 가 미결로 내려가면 실험15c 의 `IMI {12}` 유의성을 소급 철회**한다.
그건 실패가 아니라 **계측기가 자기 오차를 알게 된 것**이다.

## 한계

- **시드 3개는 분산 추정에 적다**(df=2). SD 자체의 불확실성이 크므로
  과소평가 배수도 대략적인 값이다. **"1.1배냐 2배냐" 정도만 가른다.**
- 구성 `{12}` · 부위 2개만 본다. 다른 칸의 시드 분산은 다를 수 있다
  (다만 공유 헤드 쪽은 7부위 전부의 시드 분산을 부수적으로 낸다).
- **G0 가 실패하면**(같은 시드가 재현 안 되면) 여기서 잰 "시드 분산"에는
  GPU 비결정성이 섞인다. 그 경우 배수는 **상한**으로 읽는다.
- 시드를 늘리는 것과 **예측을 시드 평균으로 앙상블하는 것**은 다른 이야기다.
  여기서는 전자(불확실성 측정)만 다룬다 — 앙상블은 성능을 올리지만
  "단일 모델의 불확실성"을 감추므로 별개 실험이다.
